# Evals & Judges

*Defining done before you ship.*

- Success metric: **the code works correctly.**
- The gap: *"I tried it a few times and it looked right"* is **not a signal you can track**.
- An eval turns that intuition into **a number you can track** as the prompt, tools, or model
  change. The rest of this module leans on it.

<details>
<summary><i>Full module text — intro</i></summary>

Your success metric is simple: the code works correctly. The agents and tools you built in the
prior modules answer correctly when you try them by hand. The gap is that "I tried it a few
times and it looked right" is not a signal you can track.

The first thing production hardening needs is a way to turn that intuition into a measurable
number that you can track as the prompt, the tools, or the model change. That is what an eval
gives you, and the rest of this module leans on it.

</details>

## First: the design document

Before any production code, write down **what you're building and how you'll know it's right**.
Usually one markdown page.

**Why it comes first:** every production layer in this module is built on it.

| Decision | States | Feeds into |
|---|---|---|
| **1. Success criteria** | What the feature must produce, for representative cases, **specific enough to grade** | The cases your **eval** is built from |
| **2. Failure handling** | Which errors production will throw, each marked **retriable or terminal**, and what the user gets when recovery fails | Your **error handling** paths |
| **3. Cost & latency budget** | Per-request budget, monthly ceiling, latency target, minimum reliability — **set before architecture** | The budget you **instrument** against, and the floor you refuse to optimise below |
| **4. Trust boundary** | Which content the agent reads that **someone else can write**, and the smallest set of actions/access needed | The input you treat as **data**, and the action you gate with a **hook** |

❌ *"summarize the thread"* — cannot be checked.
✅ *"a two-sentence summary that lists every action item and its owner"* — can be graded.

> Writing these four down once, before you build, is what keeps the layers consistent with each
> other instead of each one solving a different problem.

**Building with an agentic coding tool?** This document is what you hand it before it writes
anything. Clear success criteria + explicit constraints → fewer assumptions, and code you can
check against a document you already agreed on.

<details>
<summary><i>Full module text — Write the design document</i></summary>

Before you write any production code, write down what you are going to build and how you will
know it is right. A design document is that written record. It is a short file, usually a
single markdown page, that states the success criteria for the features, the failures the
system must survive, the cost and latency the system must stay inside, and the trust boundary
the system must defend. It is the planning step that comes before implementation, and it
exists so that you define what is correct instead of rationalizing whatever the model produces
later.

The reason the document comes first is that every production layer in this module is based on
it. The success criteria become the cases against which your eval is graded. The failures you
listed become the retriable and terminal cases your error handling must cover. The cost and
latency numbers become the budget you instrument against and the floor you refuse to optimize
below. The trust boundary becomes the input you treat as data and the action you gate with a
hook. Writing those four decisions down once, before you build, is what keeps the layers
consistent with each other instead of each one solving a different problem.

1. Success criteria name what the feature must produce. State the output for representative
cases in terms specific enough to grade, because a vague goal like "summarize the thread"
cannot be checked while "a two-sentence summary that lists every action item and its owner"
can. These criteria are what your eval set is built from, so writing them first is what makes
the eval possible.

2. Failure handling names the failures the system must survive and what it does for each. List
the errors production will throw, mark each one retriable or terminal, and say what the user
gets when a failure cannot be recovered. Deciding this on paper is what stops the first real
rate-limit response from being the moment you discover you have no error path.

3. Cost and latency budget names the ceiling the system must stay under and the reliability
floor it cannot trade away. Set hard cost and latency budgets before architecture is
determined. Write the per-request budget, the monthly cost ceiling, and the latency target,
along with the minimum reliability the design must hold. Setting these numbers before you
build is what lets you check the architecture against the budget before a line of code is
written.

4. Trust boundary names which inputs are untrusted and what the system is allowed to do. Write
down which content the agent reads that someone else can write, and the smallest set of
actions and access the feature needs to do its job. Naming the boundary on paper is what turns
least privilege into a design decision you can enforce with a hook rather than a setting you
remember to add later.

If you build an agentic coding tool, this document is also what you hand in before it writes
anything. Plan the work first and capture the result as a written artifact, then implement
against it. A tool given clear success criteria and explicit constraints makes fewer
assumptions and produces code you can check against the document you already agreed on.

</details>

## What an eval is

> **An eval works the way a thermometer does. It does not make the patient healthier. It just
> gives you a number you can trust.**

- Collect **input cases**. For each, write the **expected behaviour**.
- Run the feature on every case, **grade** each output, average the scores.
- Write the eval **before the feature** — it forces you to define success before implementation,
  instead of rationalising whatever the model produces later.

⚠️ **The score on its own is not good or bad.** First attempt scoring 2–3 out of 10 is normal.
What matters is whether the number **goes up** as you change one thing at a time.

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()

model = "claude-opus-5"

### The feature we'll grade

Straight from the design-document example: **"a two-sentence summary that lists every action
item and its owner."** Specific enough to grade.

In [3]:
DATASET = [
    {
        "thread": "Ana: the staging deploy is red. Ben: I'll roll it back now. "
                  "Ana: I'll open a postmortem doc after.",
        "expected_owners": ["Ben", "Ana"],
    },
    {
        "thread": "Chen: invoices are late again. Dee: I'll chase finance today. "
                  "Chen: ok, and I'll update the client.",
        "expected_owners": ["Dee", "Chen"],
    },
    {
        "thread": "Eve: nobody has picked up the on-call handover. "
                  "Frank: I'll take it this week and write the runbook.",
        "expected_owners": ["Frank"],
    },
    {
        "thread": "Gus: the API docs are stale. Hana: agreed. Gus: I'll refresh them Friday.",
        "expected_owners": ["Gus"],
    },
]

SYSTEM = ("Summarize the thread in exactly two sentences. "
          "List every action item and the person who owns it.")


def run_prompt(test_case):
    """Run the feature on one case and return its raw output."""
    response = client.messages.create(
        model=model,
        max_tokens=500,
        system=SYSTEM,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": test_case["thread"]}],
    )
    return next(b.text for b in response.content if b.type == "text").strip()


print(run_prompt(DATASET[0]))

**Summary:** Ana reported that the staging deploy is red, and Ben agreed to roll it back immediately. Ana will follow up by opening a postmortem document once the rollback is complete.

**Action items:**
- Roll back the staging deploy — Ben
- Open a postmortem doc (after the rollback) — Ana


## Matching the grading method to the shape of the output

The grader turns an output into a signal, usually **1–10**. Three ways — choosing wrong is where
eval effort gets wasted.

| # | Method | Works when | Cost |
|---|---|---|---|
| 1 | **Exact / string match** | The output has **one correct form** — a single label, a known value | ~free, local |
| 2 | **Code-graded check** | A **function can validate** it — valid JSON, parseable code, number in range, required field present | ~free, local |
| 3 | **LLM-as-judge** | **Open-ended quality** that no pattern captures — *"is this summary faithful?"* | **A second API call per case** |

⚠️ **The cost dimension is easy to understate.** Match and code checks run locally at
effectively zero cost — run thousands on every commit. A judge is one extra API call *per case*:
a 1,000-case eval means **1,000 extra calls every run**.

→ Many teams grade **format and structure with code on every commit**, and reserve the judge for
a **slower scheduled quality pass**.

### A code grader is often just a parse attempt

In [4]:
import json, ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10          # parses as JSON
    except json.JSONDecodeError:
        return 0           # malformed, fail the case


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


print("valid json:  ", validate_json('{"a": 1}'))
print("broken json: ", validate_json('{"a": 1'))
print("valid python:", validate_python("x = [1, 2, 3]"))
print("broken python:", validate_python("x = ["))

valid json:   10
broken json:  0
valid python: 10
broken python: 0


### Why the method has to follow the output shape

**Case A — three capital cities as a JSON array.** One run returns them in a different order
than your reference string.

In [5]:
REFERENCE = '["Canberra", "Wellington", "Suva"]'
ACTUAL    = '["Wellington", "Suva", "Canberra"]'    # same answer, different order


def exact_match(reference, actual):
    return 10 if reference.strip() == actual.strip() else 0


def code_graded_membership(reference, actual):
    """Parse both and compare as sets — order stops mattering."""
    try:
        want, got = set(json.loads(reference)), set(json.loads(actual))
    except json.JSONDecodeError:
        return 0
    return 10 if want == got else 0


print("exact match: ", exact_match(REFERENCE, ACTUAL), " <- correct answer scored ZERO")
print("code graded: ", code_graded_membership(REFERENCE, ACTUAL))

exact match:  0  <- correct answer scored ZERO
code graded:  10


**Case B — a one-paragraph rationale.** The code grader can confirm it's a non-empty string,
which is nearly worthless. Exact match is hopeless — no two good rationales are worded the same.
**Only a judge can say whether it's faithful and complete.**

> **One correct form → a match. A structural rule → a code check. Open-ended quality → a judge.**

## The grader-selection table

| Task type | Method | What it catches | Where it's unreliable |
|---|---|---|---|
| Single correct label or value | **Exact / string match** | A wrong answer when there is exactly one correct answer. Zero ambiguity, near-zero cost | Fails every valid **paraphrase or reordering** — wrong for anything open-ended |
| Structured or code output | **Code-graded check** | Invalid JSON, unparseable code, out-of-range numbers, missing required fields | Says nothing about whether the content is **good**, only that it's well-formed |
| Open-ended quality | **LLM-as-judge** | Faithfulness, instruction following, completeness, tone — things no code rule expresses | **Noisy and costly**, and produces a confident-looking number that **means nothing until calibrated** |

The judge is the only one you must **build and tune**, so it gets its own section.

<details>
<summary><i>Full module text — Grading methods</i></summary>

The grader is the part that turns an output into a measurable signal, usually a number between
one and ten. There are three ways to produce that signal, and choosing the wrong one is where
eval effort gets wasted.

1. Exact or string match works when the output has one correct form. A classifier that must
return one label, or a function that must return a known value, can be checked character by
character. It is the cheapest grader and the most brittle: any acceptable paraphrase of an
open-ended answer fails it. It is the wrong tool anytime the output can be phrased more than
one way.

2. Code-graded checks work when a function can validate the output. Valid JSON, parseable
Python, a number inside a range, a response that contains a required field: each of these is a
check you can write in code that returns a pass or a fail. The output does not have to match a
fixed string, only satisfy a rule. This method catches format and syntax failures a string
match would miss, and a human would find tedious to check by hand.

3. LLM-as-judge works for open-ended outputs where quality matters but cannot be evaluated
through pattern matching. You give a second model the output and a rubric, and it returns a
score with reasoning. This is the only method that scales questions like "is this summary
faithful?" or "did this answer follow the instructions?" because no code rule captures those.
It is also the most expensive and the noisiest, so using it when a code check would suffice
adds cost and variance for no gain.

Comparing how the same output scores under each method often makes the right choice clear.
Imagine a feature that should return the three capital cities of a region as a JSON array. One
run returns the array in a different order than your reference string. An exact match scores as
zero, because the characters do not line up, even though the answer is correct. A code grader
that parses the JSON and checks membership scores it well, because all three cities are present
and the structure is valid.

Now imagine the feature should return a one-paragraph rationale for a recommendation. The code
grader can confirm it is a non-empty string, which is nearly worthless here, and the exact match
is hopeless, because no two good rationales are worded the same. Only a judge can say whether
the rationale is faithful and complete. The method follows from the output structure: one
correct form takes a match, a structural rule takes a code check, and open-ended quality takes a
judge.

There is also a cost dimension that the table understates. An exact match and a code check run
locally and effectively cost nothing per case, so you can run thousands of them on every change.
A judge is a second model call per case, so a thousand-case eval graded by a judge is a thousand
extra API calls every time you run it. That is reasonable for a periodic full evaluation but
wasteful as a tight inner loop. Many teams grade format and structure with code on every commit
and reserve the judge for a slower, scheduled quality pass. Matching the grader to the task is
partially about signal and partially about how often you can afford to run it.

</details>

## The pipeline

Small, and the same every time: **load a dataset → run each case → grade each result → average.**

In [6]:
def run_test_case(test_case, grade):
    """Run one case through the feature, then grade the result."""
    output = run_prompt(test_case)
    score = grade(test_case, output)
    return {"output": output, "test_case": test_case, "score": score}


def run_eval(dataset, grade):
    """Run every case and report the average score."""
    results = [run_test_case(c, grade) for c in dataset]
    average = sum(r["score"] for r in results) / len(results)
    print(f"Average score: {average}")
    return results

A code grader for our success criterion — **every owner named**:

In [7]:
def grade_owners_present(test_case, output):
    """Structural check: does the summary name every expected owner?"""
    found = [o for o in test_case["expected_owners"] if o.lower() in output.lower()]
    return round(10 * len(found) / len(test_case["expected_owners"]))


results = run_eval(DATASET, grade_owners_present)

print()
for r in results:
    print(f"  score {r['score']:>2}  owners={r['test_case']['expected_owners']}")

Average score: 10.0

  score 10  owners=['Ben', 'Ana']
  score 10  owners=['Dee', 'Chen']
  score 10  owners=['Frank']
  score 10  owners=['Gus']


## Building the judge

A judge is a second model call guided by a **clear rubric**.

⚠️ **Ask for reasoning, not just a score.** Without it, models drift toward a safe middle number
— usually around **6** — regardless of actual quality. Asking for strengths, weaknesses and
reasoning **first** is what anchors the score to something specific.

The module's shape:

```python
def grade_by_model(task, solution):
    eval_prompt = f"""
    You are an expert reviewer. Evaluate the solution for the task.
    Task: {task}
    Solution: {solution}
    Return JSON with:
      "strengths":  array of 1-3 points
      "weaknesses": array of 1-3 points
      "reasoning":  a one to two sentence explanation, 50 words maximum
      "score":      a number from 1 to 10
    """
    messages = [{"role": "user", "content": eval_prompt}]
    result = chat(messages)        # returns the JSON above
    return json.loads(result)
```

Below is the same thing made robust with **structured outputs** from the prompting-craft module,
so the judge's JSON is guaranteed to parse instead of hoping `json.loads` succeeds.

In [8]:
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "strengths":  {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning":  {"type": "string"},
        "score":      {"type": "integer"},
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}


def grade_by_model(task, solution):
    """Second model call: rubric in, reasoning + score out."""
    eval_prompt = f"""You are an expert reviewer. Evaluate the solution for the task.

Task: {task}
Solution: {solution}

Score 1-10. A 10 is exactly two sentences naming every action item and its owner.
Deduct for missing owners, missing action items, or the wrong sentence count.
Give strengths and weaknesses (1-3 each) and one to two sentences of reasoning
before the score."""

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        output_config={
            "effort": "low",
            "format": {"type": "json_schema", "schema": JUDGE_SCHEMA},
        },
        messages=[{"role": "user", "content": eval_prompt}],
    )
    return json.loads(next(b.text for b in response.content if b.type == "text"))


verdict = grade_by_model(SYSTEM, results[0]["output"])
print(json.dumps(verdict, indent=2))

{
  "strengths": [
    "Summary is exactly two sentences as required",
    "Both action items are listed with clear owners (Ben: rollback, Ana: postmortem doc)",
    "Clean formatting that separates summary from action items"
  ],
  "weaknesses": [
    "Action items merely restate the summary, adding no detail such as timing or dependencies",
    "No source thread is visible to verify completeness, so unlisted items (e.g., notifying stakeholders) can't be ruled out",
    "Slight redundancy between the two sections"
  ],
  "reasoning": "The response meets the explicit format constraints: two sentences, and every stated action item paired with a named owner. The only reservation is that it cannot be confirmed exhaustive against the original thread and adds little beyond restating the summary.",
  "score": 9
}


### Grading the same run twice

Note we reuse the outputs already produced rather than re-running the feature. **Running and
grading are separate steps** — one feature run can be graded by several graders.

In [9]:
judged = []
for r in results:
    verdict = grade_by_model(SYSTEM, r["output"])
    judged.append({**r, "judge": verdict})

print(f"code-grader average: {sum(r['score'] for r in results) / len(results)}")
print(f"judge average:       {sum(j['judge']['score'] for j in judged) / len(judged)}")

print("\nper-case breakdown:")
for j in judged:
    print(f"  code={j['score']:>2}  judge={j['judge']['score']:>2}  {j['judge']['reasoning'][:70]}")

code-grader average: 10.0
judge average:       8.5

per-case breakdown:
  code=10  judge= 8  The response meets the explicit format constraints—two sentences plus 
  code=10  judge= 8  The response satisfies the explicit constraints: a two-sentence summar
  code=10  judge= 9  The summary itself is exactly two sentences and both action items are 
  code=10  judge= 9  The response meets the format spec: a two-sentence summary plus a clea


⚠️ **The per-case breakdown matters as much as the average.** A steady average can hide a change
that fixed three cases and broke three others. The average conceals it; the per-case view shows
it immediately.

## Calibrating the judge

> **Most people skip calibration, which is exactly what makes the judge untrustworthy.**

1. Start with cases **a human has already labeled**.
2. Run the judge on the same cases.
3. **Measure how often the judge agrees with the human.**

A judge that disagrees half the time produces a number that *looks* rigorous and provides no
value. If agreement is low → **fix the rubric**: tighten what each score means, add an example of
a good and a bad answer, re-measure.

In [10]:
# Human labels. Each carries the thread it summarises, so a human could check it.
THREAD = DATASET[0]["thread"]

LABELED = [
    {"thread": THREAD,
     "solution": "Ben will roll back the staging deploy. Ana will open a postmortem doc "
                 "afterwards.",
     "human": 10},   # two sentences, every owner named
    {"thread": THREAD,
     "solution": "The deploy broke and someone is looking into it.",
     "human": 2},    # no owners, no action items
    {"thread": THREAD,
     "solution": "Ben will roll back the deploy.",
     "human": 5},    # one owner named, one missing
]

TOLERANCE = 2       # within 2 points counts as agreement


def measure_agreement(judge_fn, label=""):
    agree = 0
    for case in LABELED:
        score = judge_fn(case)
        ok = abs(score - case["human"]) <= TOLERANCE
        agree += ok
        print(f"  human={case['human']:>2}  judge={score:>2}  "
              f"{'agree' if ok else 'DISAGREE'}   {case['solution'][:44]}")
    print(f"  -> agreement {label}: {agree}/{len(LABELED)} within +/-{TOLERANCE}\n")
    return agree


print("v1 rubric:")
v1 = measure_agreement(lambda c: grade_by_model(SYSTEM, c["solution"])["score"], "v1")

v1 rubric:


  human=10  judge= 5  DISAGREE   Ben will roll back the staging deploy. Ana w


  human= 2  judge= 1  agree   The deploy broke and someone is looking into


  human= 5  judge= 2  DISAGREE   Ben will roll back the deploy.
  -> agreement v1: 1/3 within +/-2



### Read the disagreement before touching anything

Low agreement is not a reason to abandon the judge — it's a **diagnosis**. Look at what the
judge said in its own `weaknesses` field:

> *"Cannot verify completeness against the original thread, which isn't shown."*

The rubric never gave the judge **the thread being summarised**, so it cannot tell a complete
summary from a partial one. It was scoring format only, and hedging on everything else.

**The fix is the rubric, not the model.** Three changes:

1. **Give it the source thread** — the thing it needs to judge completeness against.
2. **Anchor the numbers** — say what a 10, a 5 and a 1 actually mean.
3. **Show one good and one bad example**, as the module prescribes.

In [11]:
def grade_by_model_v2(case):
    """Calibrated judge: sees the source thread, anchored scale, worked examples."""
    eval_prompt = f"""You are an expert reviewer grading a thread summary.

ORIGINAL THREAD:
{case['thread']}

SUMMARY TO GRADE:
{case['solution']}

The summary must be two sentences and name every action item with its owner.

Scoring anchors:
  10 = every action item in the thread appears with the correct owner
   5 = about half the action items or owners are present
   1 = no action items or no owners identified

Example of a 10 (thread: "Ana: build is red. Ben: I'll fix it. Ana: I'll tell the team."):
  "Ben will fix the red build. Ana will notify the team."
Example of a 2 for the same thread:
  "The build is broken and someone is handling it."

Give strengths and weaknesses (1-3 each) and one to two sentences of reasoning,
then the score."""

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        output_config={
            "effort": "low",
            "format": {"type": "json_schema", "schema": JUDGE_SCHEMA},
        },
        messages=[{"role": "user", "content": eval_prompt}],
    )
    return json.loads(next(b.text for b in response.content if b.type == "text"))


print("v2 rubric (thread supplied, anchors, examples):")
v2 = measure_agreement(lambda c: grade_by_model_v2(c)["score"], "v2")

print(f"v1 -> v2 agreement: {v1}/{len(LABELED)} -> {v2}/{len(LABELED)}")

v2 rubric (thread supplied, anchors, examples):


  human=10  judge=10  agree   Ben will roll back the staging deploy. Ana w


  human= 2  judge= 1  agree   The deploy broke and someone is looking into


  human= 5  judge= 5  agree   Ben will roll back the deploy.
  -> agreement v2: 3/3 within +/-2

v1 -> v2 agreement: 1/3 -> 3/3


**That is the calibration loop, run once.** Measure agreement → read the disagreements → fix the
rubric → re-measure.

Note what we did *not* do: change the model, or accept the first number because it looked
plausible. The v1 judge produced confident-looking 6s and 2s that disagreed with human labels
on 2 of 3 cases. **A number that looks rigorous and isn't is worse than no number**, because you
would have shipped on it.

⚠️ Three cases is far too small to conclude anything real. The **procedure** is the lesson; a
production calibration set is dozens of cases, and you re-measure whenever the rubric changes.

## Coverage matters more than perfection

- A **larger set with slightly noisier automated grading** usually reveals more than a small set
  of hand-graded cases.
- **20 cases including irregular and edge inputs** will catch a break that 3 carefully chosen
  cases never exercise.
- Need more cases? **Have Claude generate them** from a small labeled starting set — then
  spot-check so the set stays honest.

> **Coverage is what catches edge cases, and coverage comes from volume.**

In [12]:
GEN_SCHEMA = {
    "type": "object",
    "properties": {
        "cases": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "thread": {"type": "string"},
                    "expected_owners": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["thread", "expected_owners"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["cases"],
    "additionalProperties": False,
}

response = client.messages.create(
    model=model,
    max_tokens=2000,
    output_config={"effort": "low",
                   "format": {"type": "json_schema", "schema": GEN_SCHEMA}},
    messages=[{"role": "user", "content":
               "Here are example test cases for a thread-summarizer eval:\n"
               f"{json.dumps(DATASET[:2], indent=2)}\n\n"
               "Generate 3 more, and make them EDGE CASES: no action items at all, "
               "one person owning several items, and an ambiguous owner."}],
)

generated = json.loads(next(b.text for b in response.content if b.type == "text"))["cases"]
for c in generated:
    print(f"  owners={c['expected_owners']}  {c['thread'][:66]}")

print("\n^ spot-check these by hand before adding them to the set")

  owners=[]  Priya: heads up, the office wifi was flaky this morning. Raj: yeah
  owners=['Tara']  Sam: three things — the migration script, the changelog, and the c
  owners=[]  Lee: someone should file the bug report for the checkout crash. Mo

^ spot-check these by hand before adding them to the set


## The loop

**Set a goal → write an initial prompt → run the eval → read where it failed → apply ONE
prompt-engineering change → run again.** Repeat the last two until the score holds.

⚠️ **Change one component at a time.** Rewrite the prompt, add two examples, and switch the model
in one pass, and when the score moves you have learned **nothing** about which change caused it.

> Slower for a single iteration, far faster over the life of the feature, because it teaches you
> what drives the score.

**A low score is information to act on.** The question isn't *whether* it failed but *why*:

| Failure looks like | Points at |
|---|---|
| Formatting wrong | The prompt's **output instructions** |
| Factually wrong on retrieved content | The **retrieval** step |
| Only breaks on long input | **Context handling** |

The eval tells you a case failed. The **per-case output tells you the category** — which is what
turns the next iteration into a targeted fix rather than a guess.

<details>
<summary><i>Full module text — The loop and calibration</i></summary>

A judge is a second model call guided by a clear rubric. What makes it usable is asking it to
provide strengths, weaknesses, and reasoning alongside the score, rather than returning the
score alone. Without that, models drift toward a safe middle number, usually around six,
regardless of the output's actual quality. Asking the judge for reasoning first is what anchors
the score to something specific.

Most people skip calibration, which is what makes the judge untrustworthy until they do it.
Start with a set of cases a human has already labeled, run the judge on the same cases, and
measure how often the judge agrees with the human. A judge that disagrees with human labels half
the time produces a number that looks rigorous but provides no value. Measuring agreement before
relying on the scores is what turns the judge from a guess into evidence you can defend. If
agreement is low, you fix the rubric: tighten what each score means, add an example of a good and
a bad answer, and re-measure.

A larger evaluation set with slightly noisier automated grading usually reveals more than a small
set of hand-graded cases. The point of an eval is to provide enough coverage to catch a
regression, not to create the perfect rubric. Twenty cases that include irregular and edge inputs
will catch a break that three carefully chosen cases never exercise. When you need more cases, you
can have Claude generate additional ones from a small, labeled starting set. You can then
spot-check the generated cases so the set stays honest. Coverage is the thing that catches edge
cases, and coverage comes from volume.

Put the three pieces together and the workflow is a loop: set a goal, write an initial prompt,
run the eval, read where it failed, apply one prompt-engineering change, and run the eval again.
You repeat the last two steps until the score holds where you need it. The eval is what tells you
a change helped instead of just feeling different.

The strategy that makes the loop work is changing one component at a time. If you rewrite the
prompt, add two examples, and switch the model all in one pass, and the score moves, you have
learned nothing about which change caused it. Move one lever, re-run, read the per-case results,
and keep the change only if the score goes up. This approach is slower for a single iteration,
but far faster than the life of the feature, because it teaches you what drives the score. The
per-case breakdown matters as much as the average. A steady average can hide a change that fixed
three cases and broke three others. The per-case view shows that immediately, while the average
conceals it.

A low score is information to act on. When a case fails, the important question is not whether it
failed, but why. A formatting failure points at the prompt's output instructions. A factual
failure on retrieved content points at the retrieval step. A failure that only appears on long
input points at context handling. The eval tells you a case failed, and the per-case output tells
you the category, which is what turns the next iteration into a targeted fix rather than a guess.

</details>

## Summary

| | |
|---|---|
| **Handles well** | Turns "looks right" into a tracked score you can defend and move **one deliberate change at a time** |
| **Adds cost or complexity** | Authoring cases and calibrating a judge is **real up-front work** before any feature ships |
| **Use a different approach** | For a single fixed-format output, a **code check alone** is enough. Skip the judge entirely |

**The design document comes first** — its success criteria *are* your eval set.